# Artificial Neural Network from Scratch with Pytorch

In [13]:
! pip3 install torch torchvision

### importing the required libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.datasets import MNIST


In [6]:
# 1. Data Preparation
# Transforms convert images to PyTorch Tensors and normalize pixels to [0, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)



In [7]:

train_dataset

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
           )

In [ ]:
# 2. Define the ANN Architecture
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten() # Flattens 28x28 images to a 784 vector
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 128),  # Hidden Layer
            nn.ReLU(),                # Activation
            nn.Linear(128, 10)        # Output Layer (10 classes for digits 0-9)
        )
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits



In [10]:
# Initialize model, loss, and optimizer
device = "cuda" if torch.cuda.is_available() else "cpu"
model = NeuralNetwork().to(device)
criterion = nn.CrossEntropyLoss() # Standard for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# 3. Training Loop
epochs = 5
print(f"Training PyTorch Model on {device}:")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        # Forward pass
        pred = model(X)
        loss = criterion(pred, y)
        # Backpropagation
        optimizer.zero_grad() # Clear gradients from previous step
        loss.backward()       # Compute new gradients
        optimizer.step()      # Update network weights
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f}")



Training PyTorch Model on cpu:
Epoch 1/5 | Loss: 0.3526
Epoch 2/5 | Loss: 0.1642
Epoch 3/5 | Loss: 0.1132
Epoch 4/5 | Loss: 0.0861
Epoch 5/5 | Loss: 0.0679


In [12]:
# 4. Evaluation Loop
model.eval()
correct = 0
total = 0

with torch.no_grad(): # Disable gradient calculation for evaluation speedup
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        outputs = model(X)
        _, predicted = torch.max(outputs, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()

print(f"\nPyTorch Test Accuracy: {(correct / total) * 100:.2f}%")


PyTorch Test Accuracy: 97.44%
